# FiLM-Conditioned Attention-MIL — NSCLC Immune Gene Prediction
**Paper:** *FiLM-Conditioned Attention-Based MIL Reveals Subtype-Specific Morphological Encoding of Antigen Presentation and T-Cell Inflammation in NSCLC*

### Before running — checklist
1. GPU accelerator enabled: **Settings > Accelerator > GPU T4 x2**
2. Your h5 feature dataset attached: **+ Add Data > Your Datasets**
3. Your metadata CSV dataset attached: **+ Add Data > Your Datasets**
4. Internet enabled (Settings > Internet > On) — needed for GitHub clone
5. Update the **PATHS** cell below to match your Kaggle dataset slugs

## Setup

In [1]:
# NOTE: EDIT THESE PATHS TO MATCH YOUR KAGGLE DATASET NAMES
# Find your dataset slug at kaggle.com/datasets/YOUR_USERNAME/DATASET_NAME

LUAD_FEATURES = "/kaggle/input/datasets/lucashuitema/tcga-luad"
LUSC_FEATURES = "/kaggle/input/datasets/lucashuitema/tcga-lusc"
METADATA_CSV  = "/kaggle/input/datasets/lucashuitema/tcga-csv/tcga-nsclc-metadata.csv"

OUTPUT_DIR    = "/kaggle/working/results"
GITHUB_REPO   = "https://github.com/LHu1t/nsclc-immune-film-mil.git"

N_FOLDS     = 5
MAX_EPOCHS  = 50
PATIENCE    = 3

FILM_ENABLED = False

print("Paths configured. Proceed to next cell.")

Paths configured. Proceed to next cell.


### Install missing dependancies and check GPU

In [2]:
# Note: torch, numpy, pandas already pre-installed on Kaggle
import subprocess
subprocess.run(["pip", "install", "h5py", "scikit-learn", "scipy", "lifelines", "-q"], check=True)
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 7.2 MB/s eta 0:00:00
Dependencies installed.


In [3]:
# Verify GPU is available
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}",
              f"| VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found. Enable GPU: Settings > Accelerator > GPU T4 x2")

PyTorch version: 2.10.0+cu128
CUDA available:  True
  GPU 0: Tesla T4 | VRAM: 15.6 GB
  GPU 1: Tesla T4 | VRAM: 15.6 GB


### Clone nsclc-immune-film-mil GitHub Repo into Kaggle

In [4]:
# Clone your GitHub repo to get the latest training script
import os
import sys

REPO_DIR = "/kaggle/working/nsclc-immune-film-mil"

if os.path.exists(REPO_DIR):
    # Pull latest changes if already cloned
    result = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print(result.stdout)
else:
    result = subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("Clone failed:", result.stderr)
        raise RuntimeError("GitHub clone failed. Check GITHUB_REPO path and that Internet is ON.")

# Add src/ to Python path so we can import train_film_mil
src_path = os.path.join(REPO_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Files in src:")
print(os.listdir(src_path))


Files in src:
['train_film_mil.py']


### Check filepaths and retrieve checkpoints

In [5]:
# Verify data paths before starting expensive training
from pathlib import Path
import pandas as pd

errors = []

# Check feature directories
for name, path in [("LUAD features", LUAD_FEATURES), ("LUSC features", LUSC_FEATURES)]:
    p = Path(path)
    if not p.exists():
        errors.append(f"{name} not found: {path}")
    else:
        h5_files = list(p.glob("*.h5"))
        print(f" {name}: {len(h5_files)} .h5 files found")
        print(f"  Example: {h5_files[0].name if h5_files else 'NONE'}")

# Check metadata CSV
if not Path(METADATA_CSV).exists():
    errors.append(f"Metadata CSV not found: {METADATA_CSV}")
else:
    df_check = pd.read_csv(METADATA_CSV, nrows=3)
    print(f"\n Metadata CSV: {pd.read_csv(METADATA_CSV).shape[0]} rows, "
          f"{len(df_check.columns)} columns")
    fpkm_cols = [c for c in pd.read_csv(METADATA_CSV, nrows=0).columns
                 if c.endswith("_fpkm_uq") or c in ("TMB", "APM", "TIS")]
    print(f"Gene/target columns found: {len(fpkm_cols)}")

if errors:
    for e in errors:
        print(e)
    raise RuntimeError("Fix the paths above before continuing.")

print("\n All data paths verified. Ready to train.")

 LUAD features: 531 .h5 files found
  Example: TCGA-49-6745-01Z-00-DX1.6bf8a38c-5e3a-4b32-8870-03bc06c0db80.h5
 LUSC features: 512 .h5 files found
  Example: TCGA-46-3765-01Z-00-DX1.f45e4e30-e60c-40e5-a0b7-4513c0c37fda.h5

 Metadata CSV: 1017 rows, 48 columns
Gene/target columns found: 38

 All data paths verified. Ready to train.


In [6]:
# Checkpoint utilities
import json
import numpy as np

CHECKPOINT_FILE = "/kaggle/working/checkpoint.json"

def _json_default(obj):
    """Convert numpy scalar/array types to native Python types for json.dump."""
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

def save_checkpoint(fold_results: list, completed_fold: int):
    """Save progress after each fold completes."""
    checkpoint = {
        "completed_fold": completed_fold,
        "fold_results":   fold_results,
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=2, default=_json_default)
    print(f"Checkpoint saved after fold {completed_fold}")

def load_checkpoint():
    """Load checkpoint if it exists, returns (fold_results, start_fold)."""
    if Path(CHECKPOINT_FILE).exists():
        with open(CHECKPOINT_FILE) as f:
            cp = json.load(f)
        start_fold = cp["completed_fold"] + 1
        print(f"Resuming from fold {start_fold} "
              f"({cp['completed_fold']} folds already completed)")
        return cp["fold_results"], start_fold
    return [], 0

print("Checkpoint utilities ready.")

Checkpoint utilities ready.


## Main Training

In [7]:
# Import training components
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from pathlib import Path

from train_film_mil import (
    load_metadata,
    FiLMDataset,
    FiLMMILModel,
    CompositeLoss,
    run_epoch,
    compute_gene_pccs,
    compute_panel_pcc,
    compute_auc,
    APM_GENES,
    TIS_GENES,
)

print("All imports successful.")

All imports successful.


In [8]:
# Load and preprocess metadata
df, gene_cols, clinical_cols, y_means, y_stds = load_metadata(METADATA_CSV)

# Build gene symbol > index map
gene_symbol_to_idx = {}
for i, g in enumerate(gene_cols):
    symbol = g.replace("_fpkm_uq", "")
    gene_symbol_to_idx[symbol] = i

feature_dirs = {"LUAD": LUAD_FEATURES, "LUSC": LUSC_FEATURES}

# Fixed 80/20 train-dev / test split
rng       = np.random.default_rng(98)
all_sids  = df["submitter_id"].unique()
test_sids = set(rng.choice(all_sids, size=int(0.2 * len(all_sids)), replace=False))
dev_sids  = [s for s in all_sids if s not in test_sids]

df_test = df[df["submitter_id"].isin(test_sids)].reset_index(drop=True)
df_dev  = df[df["submitter_id"].isin(dev_sids)].reset_index(drop=True)

print(f"Total samples : {len(df)}")
print(f"Dev set       : {len(df_dev)}")
print(f"Test set      : {len(df_test)}")
print(f"Gene targets  : {len(gene_cols)}")
#print(f"Subtypes      : LUAD={( df['cancer_type']=='LUAD').sum()}, LUSC={(df['cancer_type']=='LUSC').sum()}")

Total samples : 987
Dev set       : 790
Test set      : 197
Gene targets  : 38


In [9]:
# Main training loop with per-fold checkpointing
import os

print(f"Folds:{N_FOLDS}, Patience: {PATIENCE}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=90)

# Resume from checkpoint if available
fold_results, start_fold = load_checkpoint()

for fold, (train_idx, val_idx) in enumerate(kf.split(df_dev)):

    # Skip already-completed folds
    if fold < start_fold:
        print(f"Skipping fold {fold} (already completed)")
        continue

    print(f"\n{'='*60}")
    print(f"FOLD {fold} / {N_FOLDS - 1}")
    print(f"{'='*60}")

    df_train = df_dev.iloc[train_idx].reset_index(drop=True)
    df_val   = df_dev.iloc[val_idx].reset_index(drop=True)

    # Datasets
    train_ds = FiLMDataset(df_train, feature_dirs, gene_cols, clinical_cols,
                           n_tiles=None, deterministic=False)
    val_ds   = FiLMDataset(df_val,   feature_dirs, gene_cols, clinical_cols,
                           n_tiles=None, deterministic=True)
    test_ds  = FiLMDataset(df_test,  feature_dirs, gene_cols, clinical_cols,
                           n_tiles=None, deterministic=True)

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

    # Model, loss, optimiser
    model     = FiLMMILModel(feat_dim=1536, n_genes=len(gene_cols), use_film=FILM_ENABLED).to(device)
    loss_fn   = CompositeLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5
    )

    best_val_pcc = -np.inf
    best_weights = None
    patience_ctr = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_pcc, _, _ = run_epoch(
            model, train_loader, loss_fn, optimizer, epoch, device, training=True
        )
        val_loss, val_pcc, _, _ = run_epoch(
            model, val_loader, loss_fn, optimizer, epoch, device, training=False
        )
        scheduler.step(val_loss)

        print(f"  Epoch {epoch:3d} | "
              f"Train loss={train_loss:.4f} PCC={train_pcc:.4f} | "
              f"Val loss={val_loss:.4f} PCC={val_pcc:.4f}")

        if val_pcc > best_val_pcc:
            best_val_pcc = val_pcc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            # Save best weights to disk immediately in case of disconnect
            torch.save(best_weights, f"{OUTPUT_DIR}/fold{fold}_best_model.pt")
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    # Test set evaluation
    model.load_state_dict(best_weights)
    model.to(device)
    _, _, test_preds, test_labels = run_epoch(
        model, test_loader, loss_fn, optimizer, 999, device, training=False
    )

    gene_pccs = compute_gene_pccs(test_preds, test_labels)
    apm_pcc   = compute_panel_pcc(test_preds, test_labels, gene_cols, APM_GENES, gene_symbol_to_idx)
    tis_pcc   = compute_panel_pcc(test_preds, test_labels, gene_cols, TIS_GENES, gene_symbol_to_idx)
    apm_auc   = compute_auc(test_preds, test_labels, gene_cols, APM_GENES, gene_symbol_to_idx)
    tis_auc   = compute_auc(test_preds, test_labels, gene_cols, TIS_GENES, gene_symbol_to_idx)

    # Subtype-split evaluation
    luad_mask = np.array([r["subtype"] == "LUAD" for r in test_ds.records])
    lusc_mask = ~luad_mask
    subtype_results = {}
    for name, mask in [("LUAD", luad_mask), ("LUSC", lusc_mask)]:
        if mask.sum() == 0:
            continue
        p, l = test_preds[mask], test_labels[mask]
        subtype_results[name] = {
            "APM_PCC": compute_panel_pcc(p, l, gene_cols, APM_GENES, gene_symbol_to_idx),
            "TIS_PCC": compute_panel_pcc(p, l, gene_cols, TIS_GENES, gene_symbol_to_idx),
            "APM_AUC": compute_auc(p, l, gene_cols, APM_GENES, gene_symbol_to_idx),
            "TIS_AUC": compute_auc(p, l, gene_cols, TIS_GENES, gene_symbol_to_idx),
            "n":       int(mask.sum()),
        }

    fold_result = {
        "fold":         fold,
        "best_val_pcc": float(best_val_pcc),
        "APM_PCC":      float(apm_pcc),
        "TIS_PCC":      float(tis_pcc),
        "APM_AUC":      float(apm_auc),
        "TIS_AUC":      float(tis_auc),
        "gene_pccs":    {g.replace("_fpkm_uq", ""): float(gene_pccs[i])
                         for i, g in enumerate(gene_cols)},
        "subtype":      subtype_results,
    }
    fold_results.append(fold_result)

    print(f"\nFold {fold} Results:")
    print(f"  APM  PCC={apm_pcc:.4f}  AUC={apm_auc:.4f}")
    print(f"  TIS  PCC={tis_pcc:.4f}  AUC={tis_auc:.4f}")
    for name, res in subtype_results.items():
        print(f"  {name} (n={res['n']}): APM={res['APM_PCC']:.4f}, TIS={res['TIS_PCC']:.4f}")

    # Save checkpoint after this fold completes
    save_checkpoint(fold_results, fold)

print("\n All folds complete.")

2026-08-01 14:31:12,814 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-08-01 14:31:12,819 [INFO]   LUSC feature index: 478 unique patient barcodes


Folds:5, Patience: 3

FOLD 0 / 4


2026-08-01 14:31:13,361 [INFO] Dataset: 582 matched slides (LUAD: 299, LUSC: 283)
2026-08-01 14:31:13,366 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-08-01 14:31:13,370 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-08-01 14:31:13,484 [INFO] Dataset: 144 matched slides (LUAD: 72, LUSC: 72)
2026-08-01 14:31:13,488 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-08-01 14:31:13,492 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-08-01 14:31:13,641 [INFO] Dataset: 186 matched slides (LUAD: 82, LUSC: 104)


Train: 582 | Val: 144 | Test: 186
  Epoch   1 | Train loss=0.9968 PCC=0.1034 | Val loss=0.9839 PCC=0.2071
  Epoch   2 | Train loss=0.8929 PCC=0.3012 | Val loss=0.9368 PCC=0.3133
  Epoch   3 | Train loss=0.8090 PCC=0.4210 | Val loss=0.8896 PCC=0.3922
  Epoch   4 | Train loss=0.7271 PCC=0.5112 | Val loss=0.8360 PCC=0.4458
  Epoch   5 | Train loss=0.6562 PCC=0.5762 | Val loss=0.8185 PCC=0.4609
  Epoch   6 | Train loss=0.5967 PCC=0.6248 | Val loss=0.8170 PCC=0.4674
  Epoch   7 | Train loss=0.5679 PCC=0.6468 | Val loss=0.8091 PCC=0.4742
  Epoch   8 | Train loss=0.5111 PCC=0.6916 | Val loss=0.7868 PCC=0.4944
  Epoch   9 | Train loss=0.4817 PCC=0.7117 | Val loss=0.8078 PCC=0.4830
  Epoch  10 | Train loss=0.4481 PCC=0.7359 | Val loss=0.8169 PCC=0.4656
  Epoch  11 | Train loss=0.4144 PCC=0.7593 | Val loss=0.8090 PCC=0.4808
  Early stopping at epoch 11


2026-08-01 15:17:50,425 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-08-01 15:17:50,447 [INFO]   LUSC feature index: 478 unique patient barcodes


...

Train: 576 | Val: 150 | Test: 186
  Epoch   1 | Train loss=1.0556 PCC=0.0748 | Val loss=0.9706 PCC=0.2093
  Epoch   2 | Train loss=0.9156 PCC=0.2649 | Val loss=0.9232 PCC=0.3091
  Epoch   3 | Train loss=0.8310 PCC=0.3986 | Val loss=0.8607 PCC=0.4025
  Epoch   4 | Train loss=0.7459 PCC=0.4937 | Val loss=0.8192 PCC=0.4396
  Epoch   5 | Train loss=0.6732 PCC=0.5619 | Val loss=0.8274 PCC=0.4532
  Epoch   6 | Train loss=0.6257 PCC=0.6056 | Val loss=0.7894 PCC=0.4718
  Epoch   7 | Train loss=0.5771 PCC=0.6435 | Val loss=0.7897 PCC=0.4692
  Epoch   8 | Train loss=0.5370 PCC=0.6739 | Val loss=0.7996 PCC=0.4595
  Epoch   9 | Train loss=0.5117 PCC=0.6938 | Val loss=0.8235 PCC=0.4528
  Early stopping at epoch 9


2026-08-01 17:29:34,224 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-08-01 17:29:34,228 [INFO]   LUSC feature index: 478 unique patient barcodes



Fold 3 Results:
  APM  PCC=0.5336  AUC=0.8035
  TIS  PCC=0.6870  AUC=0.8855
  LUAD (n=82): APM=0.5204, TIS=0.7618
  LUSC (n=104): APM=0.4967, TIS=0.6311
Checkpoint saved after fold 3

FOLD 4 / 4


2026-08-01 17:29:34,733 [INFO] Dataset: 580 matched slides (LUAD: 294, LUSC: 286)
2026-08-01 17:29:34,739 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-08-01 17:29:34,743 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-08-01 17:29:34,856 [INFO] Dataset: 146 matched slides (LUAD: 77, LUSC: 69)
2026-08-01 17:29:34,860 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-08-01 17:29:34,863 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-08-01 17:29:35,014 [INFO] Dataset: 186 matched slides (LUAD: 82, LUSC: 104)


Train: 580 | Val: 146 | Test: 186
  Epoch   1 | Train loss=1.2365 PCC=0.0592 | Val loss=0.9011 PCC=0.1943
  Epoch   2 | Train loss=0.9656 PCC=0.2114 | Val loss=0.8912 PCC=0.2206
  Epoch   3 | Train loss=0.9031 PCC=0.3231 | Val loss=0.8587 PCC=0.2866
  Epoch   4 | Train loss=0.8227 PCC=0.4303 | Val loss=0.8362 PCC=0.3353
  Epoch   5 | Train loss=0.7554 PCC=0.5017 | Val loss=0.8272 PCC=0.3466
  Epoch   6 | Train loss=0.6820 PCC=0.5691 | Val loss=0.8100 PCC=0.3957
  Epoch   7 | Train loss=0.6229 PCC=0.6169 | Val loss=0.8091 PCC=0.3845
  Epoch   8 | Train loss=0.5690 PCC=0.6579 | Val loss=0.7966 PCC=0.3996
  Epoch   9 | Train loss=0.5337 PCC=0.6844 | Val loss=0.8256 PCC=0.4148
  Epoch  10 | Train loss=0.4891 PCC=0.7162 | Val loss=0.8138 PCC=0.3941
  Epoch  11 | Train loss=0.4563 PCC=0.7389 | Val loss=0.8012 PCC=0.4060
  Epoch  12 | Train loss=0.4270 PCC=0.7592 | Val loss=0.8202 PCC=0.4015
  Early stopping at epoch 12

Fold 4 Results:
  APM  PCC=0.5816  AUC=0.8163
  TIS  PCC=0.6957  AUC=0.8

In [10]:
# Cross-validation summary
print("CROSS-VALIDATION SUMMARY")

for metric in ["APM_PCC", "TIS_PCC", "APM_AUC", "TIS_AUC"]:
    vals = [r[metric] for r in fold_results]
    print(f"  {metric:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print()
for subtype in ["LUAD", "LUSC"]:
    print(f"  {subtype}")
    for metric in ["APM_PCC", "TIS_PCC", "APM_AUC", "TIS_AUC"]:
        vals = [r["subtype"][subtype][metric]
                for r in fold_results if subtype in r["subtype"]]
        if vals:
            print(f"  {metric:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

# Gene-level summary — top and bottom 5 across folds
print("\n Gene-level PCC (mean across folds)")
all_gene_symbols = list(fold_results[0]["gene_pccs"].keys())
mean_gene_pccs = {
    g: np.mean([r["gene_pccs"][g] for r in fold_results])
    for g in all_gene_symbols
}
sorted_genes = sorted(mean_gene_pccs.items(), key=lambda x: x[1], reverse=True)
print("  Top 5 genes:")
for g, r in sorted_genes[:5]:
    print(f"    {g:15s}: {r:.4f}")
print("  Bottom 5 genes:")
for g, r in sorted_genes[-5:]:
    print(f"    {g:15s}: {r:.4f}")

CROSS-VALIDATION SUMMARY
  APM_PCC     : 0.5662 ± 0.0188
  TIS_PCC     : 0.6934 ± 0.0094
  APM_AUC     : 0.8071 ± 0.0123
  TIS_AUC     : 0.8811 ± 0.0076

  LUAD
  APM_PCC     : 0.5329 ± 0.0288
  TIS_PCC     : 0.7281 ± 0.0218
  APM_AUC     : 0.7742 ± 0.0156
  TIS_AUC     : 0.9468 ± 0.0157
  LUSC
  APM_PCC     : 0.5476 ± 0.0300
  TIS_PCC     : 0.6644 ± 0.0202
  APM_AUC     : 0.7888 ± 0.0147
  TIS_AUC     : 0.8037 ± 0.0122

 Gene-level PCC (mean across folds)
  Top 5 genes:
    CCL5           : 0.6591
    TIS            : 0.6580
    CD8A           : 0.6496
    NKG7           : 0.6428
    CXCL9          : 0.6409
  Bottom 5 genes:
    TMB            : 0.3051
    PSMB6          : 0.2999
    PSMB5          : 0.2916
    CALR           : 0.2164
    ERAP2          : 0.1660


In [11]:
# Save final results JSON
import json

results_path = f"{OUTPUT_DIR}/results.json"
with open(results_path, "w") as f:
    json.dump(fold_results, f, indent=2, default=_json_default)

print(f"Results saved to: {results_path}")
print("\nModel weights saved:")
for pt in Path(OUTPUT_DIR).glob("*.pt"):
    print(f"  {pt.name}  ({pt.stat().st_size / 1e6:.1f} MB)")

Results saved to: /kaggle/working/results/results.json

Model weights saved:
  fold0_best_model.pt  (7.5 MB)
  fold3_best_model.pt  (7.5 MB)
  fold2_best_model.pt  (7.5 MB)
  fold1_best_model.pt  (7.5 MB)
  fold4_best_model.pt  (7.5 MB)


## Reproducibility

In [12]:
# Patient ID lists per split + leakage check
import os, json
import numpy as np
from sklearn.model_selection import KFold

EXPORT_DIR = f"{OUTPUT_DIR}/paper_artifacts"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Re-derive the exact per-fold train/val split
kf_check = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_splits = list(kf_check.split(df_dev))
assert len(fold_splits) == N_FOLDS

test_sids_set = set(df_test["submitter_id"])
split_record = {"test": sorted(test_sids_set)}

leakage_found = False
for fold, (train_idx, val_idx) in enumerate(fold_splits):
    train_sids = set(df_dev.iloc[train_idx]["submitter_id"])
    val_sids   = set(df_dev.iloc[val_idx]["submitter_id"])

    split_record[f"fold{fold}_train"] = sorted(train_sids)
    split_record[f"fold{fold}_val"]   = sorted(val_sids)

    # Leakage checks
    tt = train_sids & test_sids_set
    vt = val_sids & test_sids_set
    tv = train_sids & val_sids
    if tt or vt or tv:
        leakage_found = True
        print(f"Fold {fold}: train∩test={len(tt)} val∩test={len(vt)} train∩val={len(tv)}")

if not leakage_found:
    print("No patient-level overlap between train/val/test in any fold.")
else:
    print("LEAKAGE DETECTED — see above. Do not report results until resolved.")

with open(f"{EXPORT_DIR}/patient_id_splits.json", "w") as f:
    json.dump(split_record, f, indent=2)

print(f"Saved patient ID splits: {EXPORT_DIR}/patient_id_splits.json")
print(f"Test: {len(test_sids_set)} | "
      f"Fold0 train/val: {len(fold_splits[0][0])}/{len(fold_splits[0][1])}")

No patient-level overlap between train/val/test in any fold.
Saved patient ID splits: /kaggle/working/results/paper_artifacts/patient_id_splits.json
  Test: 197 | Fold0 train/val: 632/158


In [13]:
# Raw predictions + labels per fold (reloaded from checkpoints)
import torch
from torch.utils.data import DataLoader
from pathlib import Path

# Fixed test set — identical across every fold by construction
test_ds = FiLMDataset(df_test, feature_dirs, gene_cols, clinical_cols,
                       n_tiles=None, deterministic=True)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)

test_sids_ordered     = [r["sid"] for r in test_ds.records]      # row order guarantee
test_subtypes_ordered = [r["subtype"] for r in test_ds.records]  # (shuffle=False)

all_fold_preds = {}   # fold -> (n_test, n_genes) array
all_fold_labels = None

for fold in range(N_FOLDS):
    ckpt_path = Path(OUTPUT_DIR) / f"fold{fold}_best_model.pt"
    if not ckpt_path.exists():
        print(f"Fold {fold}: checkpoint not found, skipping (not completed yet?)")
        continue

    model = FiLMMILModel(feat_dim=1536, n_genes=len(gene_cols), use_film=FILM_ENABLED).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    loss_fn   = CompositeLoss()
    optimizer = torch.optim.Adam(model.parameters())  # unused (training=False), needed for signature

    _, _, test_preds, test_labels = run_epoch(
        model, test_loader, loss_fn, optimizer, 999, device, training=False
    )
    all_fold_preds[fold] = test_preds
    all_fold_labels = test_labels  # identical every fold (same fixed test set)

    np.savez(
        f"{EXPORT_DIR}/fold{fold}_test_predictions.npz",
        preds=test_preds, labels=test_labels,
        submitter_id=np.array(test_sids_ordered),
        subtype=np.array(test_subtypes_ordered),
        gene_cols=np.array(gene_cols),
    )
    print(f"  Fold {fold}: saved raw predictions {test_preds.shape} "
          f"to fold{fold}_test_predictions.npz")

# Ensemble prediction (mean across folds, same fixed test set) — your
# headline number, since all folds evaluate the identical test set
ensemble_preds = np.mean(list(all_fold_preds.values()), axis=0)
np.savez(
    f"{EXPORT_DIR}/ensemble_test_predictions.npz",
    preds=ensemble_preds, labels=all_fold_labels,
    submitter_id=np.array(test_sids_ordered),
    subtype=np.array(test_subtypes_ordered),
    gene_cols=np.array(gene_cols),
)
print(f"Ensemble predictions saved ({len(all_fold_preds)} folds averaged)")

# Tidy long-format CSV for quick plotting (predicted vs actual, per gene, per patient)
import pandas as pd
rows = []
for i, sid in enumerate(test_sids_ordered):
    for g_idx, g in enumerate(gene_cols):
        rows.append({
            "submitter_id": sid,
            "subtype": test_subtypes_ordered[i],
            "gene": g.replace("_fpkm_uq", ""),
            "predicted": ensemble_preds[i, g_idx],
            "actual": all_fold_labels[i, g_idx],
        })
pd.DataFrame(rows).to_csv(f"{EXPORT_DIR}/ensemble_predictions_long.csv", index=False)
print(f"Long-format CSV for scatter plots saved ({len(rows)} rows)")

2026-08-01 18:26:17,718 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-08-01 18:26:17,723 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-08-01 18:26:17,893 [INFO] Dataset: 186 matched slides (LUAD: 82, LUSC: 104)


  Fold 0: saved raw predictions (186, 38) to fold0_test_predictions.npz
  Fold 1: saved raw predictions (186, 38) to fold1_test_predictions.npz
  Fold 2: saved raw predictions (186, 38) to fold2_test_predictions.npz
  Fold 3: saved raw predictions (186, 38) to fold3_test_predictions.npz
  Fold 4: saved raw predictions (186, 38) to fold4_test_predictions.npz
Ensemble predictions saved (5 folds averaged)
Long-format CSV for scatter plots saved (7068 rows)


In [14]:
# Bootstrap CIs + permutation p-values
N_BOOT = 2000
N_PERM = 2000
rng = np.random.default_rng(0)

def bootstrap_ci(preds, labels, gene_list, n_boot=N_BOOT):
    n = preds.shape[0]
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)  # resample patients with replacement
        vals.append(compute_panel_pcc(preds[idx], labels[idx], gene_cols, gene_list, gene_symbol_to_idx))
    vals = np.array(vals)
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def permutation_pvalue(preds, labels, gene_list, n_perm=N_PERM):
    observed = compute_panel_pcc(preds, labels, gene_cols, gene_list, gene_symbol_to_idx)
    n = preds.shape[0]
    null_vals = np.empty(n_perm)
    for i in range(n_perm):
        perm_idx = rng.permutation(n)
        null_vals[i] = compute_panel_pcc(preds, labels[perm_idx], gene_cols, gene_list, gene_symbol_to_idx)
    p = (np.sum(null_vals >= observed) + 1) / (n_perm + 1)
    return float(observed), float(p)

stats_summary = {}
targets = {"APM": APM_GENES, "TIS": TIS_GENES}

# Per-fold + ensemble
sources = {**{f"fold{f}": p for f, p in all_fold_preds.items()}, "ensemble": ensemble_preds}
for name, preds in sources.items():
    stats_summary[name] = {}
    for panel, genes in targets.items():
        mean_pcc, lo, hi = bootstrap_ci(preds, all_fold_labels, genes)
        observed, p = permutation_pvalue(preds, all_fold_labels, genes)
        stats_summary[name][panel] = {
            "PCC": observed, "bootstrap_mean": mean_pcc,
            "CI95_low": lo, "CI95_high": hi, "perm_p": p,
        }
        print(f"{name:10s} {panel}: PCC={observed:.4f} 95%CI=[{lo:.4f},{hi:.4f}] p={p:.4f}")

with open(f"{EXPORT_DIR}/bootstrap_permutation_stats.json", "w") as f:
    json.dump(stats_summary, f, indent=2)
print(f"Saved: {EXPORT_DIR}/bootstrap_permutation_stats.json")

fold0      APM: PCC=0.5689 95%CI=[0.4626,0.6611] p=0.0005
fold0      TIS: PCC=0.7027 95%CI=[0.6079,0.7783] p=0.0005
fold1      APM: PCC=0.5867 95%CI=[0.4921,0.6722] p=0.0005
fold1      TIS: PCC=0.7030 95%CI=[0.6159,0.7765] p=0.0005
fold2      APM: PCC=0.5600 95%CI=[0.4565,0.6449] p=0.0005
fold2      TIS: PCC=0.6786 95%CI=[0.5892,0.7549] p=0.0005
fold3      APM: PCC=0.5336 95%CI=[0.4263,0.6313] p=0.0005
fold3      TIS: PCC=0.6870 95%CI=[0.5810,0.7703] p=0.0005
fold4      APM: PCC=0.5816 95%CI=[0.4710,0.6774] p=0.0005
fold4      TIS: PCC=0.6957 95%CI=[0.6075,0.7740] p=0.0005
ensemble   APM: PCC=0.5964 95%CI=[0.5013,0.6830] p=0.0005
ensemble   TIS: PCC=0.7260 95%CI=[0.6447,0.7962] p=0.0005
Saved: /kaggle/working/results/paper_artifacts/bootstrap_permutation_stats.json


In [18]:
# y_means/y_stds, gene order, model & training config
from train_film_mil import SUBTYPE_MAP

config = {
    "gene_cols": list(gene_cols),                 # exact order model expects/outputs
    "clinical_cols": list(clinical_cols),
    "y_means": {g: float(y_means[g]) for g in gene_cols} if hasattr(y_means, "__getitem__") else list(map(float, y_means)),
    "y_stds":  {g: float(y_stds[g])  for g in gene_cols} if hasattr(y_stds, "__getitem__") else list(map(float, y_stds)),
    "SUBTYPE_MAP": SUBTYPE_MAP,
    "APM_GENES": APM_GENES,
    "TIS_GENES": TIS_GENES,
    "model": {
        "architecture": "FiLMMILModel",
        "feat_dim": 1536,
        "n_genes": len(gene_cols),
    },
    "training": {
        "n_folds": N_FOLDS,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "use_film": FILM_ENABLED,
        "optimizer": "Adam", "lr": 1e-4, "weight_decay": 1e-5,
        "scheduler": "ReduceLROnPlateau", "scheduler_patience": 5, "scheduler_factor": 0.5,
        "batch_size": 1,
        "loss": "CompositeLoss (per-sample MSE + batch-wise composite every 16 slides)",
        "gradient_clip_norm": 1.0,
        "dev_test_split_seed": 42, "kfold_random_state": 42,
    },
    "feature_extractor": {
        "name": "UNI2-h Pretrained vision backbone (ViT-H/14 via DINOv2)",                    # ⚠ FILL IN: exact checkpoint/version/revision you used
        "patch_size_px": "256 x 256",
        "magnification": "20x",
    },
}

with open(f"{EXPORT_DIR}/model_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved model_config.json")

Saved model_config.json


In [19]:
# Cohort characteristics table
candidate_cols = {
    "age": ["age_years"],
    "sex": ["demographic.gender"],
    "stage": ["diagnoses.0.ajcc_pathologic_stage"],
}
found_cols = {}
for label, candidates in candidate_cols.items():
    for c in candidates:
        if c in df.columns:
            found_cols[label] = c
            break

print("Columns found for Table 1:", found_cols)
missing = [k for k in candidate_cols if k not in found_cols]
if missing:
    print(f"Not found in metadata CSV: {missing}")

splits = {"Test": df_test, "Dev (train+val)": df_dev, "Full cohort": df}
table1_rows = []
for split_name, split_df in splits.items():
    for subtype in ["LUAD", "LUSC"]:
        sub = split_df[split_df["cancer_type"] == subtype] if "cancer_type" in split_df.columns else split_df
        row = {"split": split_name, "subtype": subtype, "n": len(sub)}
        if "age" in found_cols:
            row["age_mean"] = float(sub[found_cols["age"]].mean())
            row["age_sd"]   = float(sub[found_cols["age"]].std())
        if "sex" in found_cols:
            row["sex_counts"] = sub[found_cols["sex"]].value_counts().to_dict()
        if "stage" in found_cols:
            row["stage_counts"] = sub[found_cols["stage"]].value_counts().to_dict()
        table1_rows.append(row)

table1_df = pd.DataFrame(table1_rows)
table1_df.to_csv(f"{EXPORT_DIR}/table1_cohort_characteristics.csv", index=False)
print(table1_df)
print(f"Saved: {EXPORT_DIR}/table1_cohort_characteristics.csv")

Columns found for Table 1: {'age': 'age_years', 'sex': 'demographic.gender', 'stage': 'diagnoses.0.ajcc_pathologic_stage'}
             split subtype    n   age_mean    age_sd  \
0             Test    LUAD  197  67.892334  8.665685   
1             Test    LUSC  197  67.892334  8.665685   
2  Dev (train+val)    LUAD  790  66.588608  9.100537   
3  Dev (train+val)    LUSC  790  66.588608  9.100537   
4      Full cohort    LUAD  987  66.848825  9.026176   
5      Full cohort    LUSC  987  66.848825  9.026176   

                     sex_counts  \
0   {'male': 126, 'female': 71}   
1   {'male': 126, 'female': 71}   
2  {'male': 462, 'female': 328}   
3  {'male': 462, 'female': 328}   
4  {'male': 588, 'female': 399}   
5  {'male': 588, 'female': 399}   

                                        stage_counts  
0  {'Stage I': 85, 'Stage II': 39, 'Stage III': 2...  
1  {'Stage I': 85, 'Stage II': 39, 'Stage III': 2...  
2  {'Stage I': 329, 'Stage II': 186, 'Stage III':...  
3  {'Stage I': 329